# Notebook 3: MongoDB Atlas, PyMongo, Aggregation and Query Optimisation

In [125]:
import pandas as pd
import numpy as np
import zipfile
import os
import json
from pprint import pprint

print("MongoDB notebook started successfully")

MongoDB notebook started successfully


##1. Extract Dataset

In [126]:
zip_path = "northstar_dataset.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("northstar_dataset")

print("Dataset extracted successfully")

for root, dirs, files in os.walk("northstar_dataset"):
    print("Folder:", root)
    print("Files:", files)
    print("-" * 50)

Dataset extracted successfully
Folder: northstar_dataset
Files: []
--------------------------------------------------
Folder: northstar_dataset/northstar_dataset
Files: ['customers.csv', 'data_dictionary.csv', 'complaints.csv', 'app_events.csv', 'deliveries.csv', 'hubs.csv', 'incidents.csv', 'drivers.csv', 'vehicles.csv', 'orders.csv', 'README.txt']
--------------------------------------------------


## 2. Load and Prepare Dataset

In [127]:
data_path = "northstar_dataset/northstar_dataset/"

customers = pd.read_csv(data_path + "customers.csv")
orders = pd.read_csv(data_path + "orders.csv")
deliveries = pd.read_csv(data_path + "deliveries.csv")
drivers = pd.read_csv(data_path + "drivers.csv")
vehicles = pd.read_csv(data_path + "vehicles.csv")
hubs = pd.read_csv(data_path + "hubs.csv")
incidents = pd.read_csv(data_path + "incidents.csv")
complaints = pd.read_csv(data_path + "complaints.csv")
app_events = pd.read_csv(data_path + "app_events.csv")

print("customers:", customers.shape)
print("orders:", orders.shape)
print("deliveries:", deliveries.shape)
print("drivers:", drivers.shape)
print("vehicles:", vehicles.shape)
print("hubs:", hubs.shape)
print("incidents:", incidents.shape)
print("complaints:", complaints.shape)
print("app_events:", app_events.shape)

customers: (650, 9)
orders: (1250, 11)
deliveries: (950, 13)
drivers: (170, 8)
vehicles: (120, 8)
hubs: (8, 5)
incidents: (280, 7)
complaints: (320, 10)
app_events: (640, 10)


## 3. Create Risk Flags

In [128]:
deliveries["failed_delivery_flag"] = np.where(
    deliveries["delivery_status"].str.lower() == "failed", 1, 0
)

deliveries["delayed_delivery_flag"] = np.where(
    deliveries["delivery_status"].str.lower() == "delayed", 1, 0
)

deliveries["route_override_flag"] = np.where(
    deliveries["manual_route_override_count"] > 0, 1, 0
)

app_events["app_failure_flag"] = np.where(
    app_events["success_flag"] == 0, 1, 0
)

print("Risk flags created successfully")

Risk flags created successfully


## 4. Create MongoDB-Style Service Case Documents

In [129]:
# Convert reference tables to dictionaries for fast lookup
customers_dict = customers.set_index("customer_id").to_dict("index")
drivers_dict = drivers.set_index("driver_id").to_dict("index")
vehicles_dict = vehicles.set_index("vehicle_id").to_dict("index")
hubs_dict = hubs.set_index("hub_id").to_dict("index")

service_cases = []

for _, order in orders.iterrows():
    order_id = order["order_id"]
    customer_id = order["customer_id"]

    delivery_rows = deliveries[deliveries["order_id"] == order_id]

    if len(delivery_rows) > 0:
        delivery = delivery_rows.iloc[0].to_dict()
        delivery_id = delivery.get("delivery_id")
        driver_id = delivery.get("driver_id")
        vehicle_id = delivery.get("vehicle_id")
        hub_id = delivery.get("hub_id")
    else:
        delivery = {}
        delivery_id = None
        driver_id = None
        vehicle_id = None
        hub_id = None

    customer = customers_dict.get(customer_id, {})
    driver = drivers_dict.get(driver_id, {}) if driver_id else {}
    vehicle = vehicles_dict.get(vehicle_id, {}) if vehicle_id else {}
    hub = hubs_dict.get(hub_id, {}) if hub_id else {}

    complaint_list = complaints[complaints["order_id"] == order_id].to_dict("records")
    incident_list = incidents[incidents["delivery_id"] == delivery_id].to_dict("records") if delivery_id else []
    app_event_list = app_events[app_events["order_id"] == order_id].to_dict("records")

    delivery_status = str(delivery.get("delivery_status", "")).lower()
    rating = delivery.get("customer_rating_post_delivery", None)
    route_override_count = delivery.get("manual_route_override_count", 0)

    service_case = {
        "order_id": order_id,
        "customer": {
            "customer_id": customer_id,
            "customer_type": customer.get("customer_type"),
            "home_zone": customer.get("home_zone"),
            "loyalty_tier": customer.get("loyalty_tier")
        },
        "order": order.to_dict(),
        "delivery": delivery,
        "driver": driver,
        "vehicle": vehicle,
        "hub": hub,
        "complaints": complaint_list,
        "incidents": incident_list,
        "app_events": app_event_list,
        "risk_flags": {
            "failed_delivery": delivery_status == "failed",
            "delayed_delivery": delivery_status == "delayed",
            "has_complaint": len(complaint_list) > 0,
            "has_incident": len(incident_list) > 0,
            "route_override_used": route_override_count > 0,
            "low_rating": rating is not None and rating < 3
        }
    }

    service_cases.append(service_case)

print("Total service case documents created:", len(service_cases))

Total service case documents created: 1250


## 5. Example Service Case Document

In [130]:
pprint(service_cases[0])

{'app_events': [{'api_latency_ms': 204,
                 'app_failure_flag': 0,
                 'customer_id': 'C0112',
                 'device_type': 'Android',
                 'event_id': 'AE00503',
                 'event_timestamp': '2024-08-02 12:35:00',
                 'event_type': 'delivery_instruction_update',
                 'order_id': 'O00001',
                 'session_id': 'S44209',
                 'success_flag': 1,
                 'zone_context': 'Riverside'}],
 'complaints': [],
 'customer': {'customer_id': 'C0292',
              'customer_type': 'Consumer',
              'home_zone': 'South',
              'loyalty_tier': None},
 'delivery': {'customer_rating_post_delivery': 4.29,
              'delayed_delivery_flag': 0,
              'delivery_completed_at': '2024-08-20 18:52:56.172161',
              'delivery_id': 'DL00937',
              'delivery_status': 'OnTime',
              'dispatch_time': '2024-08-20 16:29:00',
              'driver_id': 'D047',
  

## 6. Save Sample JSON File

In [131]:
with open("service_cases_sample.json", "w") as f:
    json.dump(service_cases[:50], f, indent=4, default=str)

print("Sample service case JSON file saved successfully")

Sample service case JSON file saved successfully


 7. CRUD Operations Demonstration

In [132]:
# CREATE operation: add a new service case

new_service_case = {
    "order_id": "O9999",
    "customer": {
        "customer_id": "C999",
        "customer_type": "Business",
        "home_zone": "Central"
    },
    "order": {
        "service_type": "Parcel",
        "priority_level": "High",
        "order_value": 99.50
    },
    "delivery": {
        "delivery_id": "DL9999",
        "delivery_status": "Failed",
        "manual_route_override_count": 2,
        "customer_rating_post_delivery": 2.5
    },
    "complaints": [
        {
            "complaint_id": "CMP9999",
            "complaint_type": "Late delivery",
            "severity": "High",
            "status": "Open",
            "compensation_amount": 0
        }
    ],
    "incidents": [],
    "app_events": [],
    "risk_flags": {
        "failed_delivery": True,
        "delayed_delivery": False,
        "has_complaint": True,
        "has_incident": False,
        "route_override_used": True,
        "low_rating": True
    }
}

service_cases.append(new_service_case)

print("CREATE operation completed")
print("Total service cases after insert:", len(service_cases))

CREATE operation completed
Total service cases after insert: 1251


In [133]:
# READ operation: find failed delivery cases

failed_cases = [
    case for case in service_cases
    if case["risk_flags"]["failed_delivery"] == True
]

print("READ operation completed")
print("Number of failed service cases:", len(failed_cases))

pprint(failed_cases[0])

READ operation completed
Number of failed service cases: 133
{'app_events': [],
 'complaints': [{'channel': 'Phone',
                 'compensation_amount': 0.0,
                 'complaint_id': 'CP0124',
                 'complaint_type': 'SupportExperience',
                 'created_at': '2025-10-16 04:59:00',
                 'customer_id': 'C0077',
                 'order_id': 'O00023',
                 'resolution_days': 7,
                 'severity': 'Low',
                 'status': 'Resolved'}],
 'customer': {'customer_id': 'C0077',
              'customer_type': 'SME',
              'home_zone': 'NORTH',
              'loyalty_tier': None},
 'delivery': {'customer_rating_post_delivery': 2.38,
              'delayed_delivery_flag': 0,
              'delivery_completed_at': '2025-10-06 04:17:27.447541',
              'delivery_id': 'DL00831',
              'delivery_status': 'Failed',
              'dispatch_time': '2025-10-05 05:51:00',
              'driver_id': 'D133',
    

In [134]:
# UPDATE operation: update complaint status for test order O9999

for case in service_cases:
    if case["order_id"] == "O9999":
        case["complaints"][0]["status"] = "Resolved"
        case["complaints"][0]["compensation_amount"] = 10.00

print("UPDATE operation completed")

for case in service_cases:
    if case["order_id"] == "O9999":
        pprint(case)

UPDATE operation completed
{'app_events': [],
 'complaints': [{'compensation_amount': 10.0,
                 'complaint_id': 'CMP9999',
                 'complaint_type': 'Late delivery',
                 'severity': 'High',
                 'status': 'Resolved'}],
 'customer': {'customer_id': 'C999',
              'customer_type': 'Business',
              'home_zone': 'Central'},
 'delivery': {'customer_rating_post_delivery': 2.5,
              'delivery_id': 'DL9999',
              'delivery_status': 'Failed',
              'manual_route_override_count': 2},
 'incidents': [],
 'order': {'order_value': 99.5,
           'priority_level': 'High',
           'service_type': 'Parcel'},
 'order_id': 'O9999',
 'risk_flags': {'delayed_delivery': False,
                'failed_delivery': True,
                'has_complaint': True,
                'has_incident': False,
                'low_rating': True,
                'route_override_used': True}}


In [135]:
# DELETE operation: remove the test service case

service_cases = [
    case for case in service_cases
    if case["order_id"] != "O9999"
]

print("DELETE operation completed")
print("Total service cases after delete:", len(service_cases))

DELETE operation completed
Total service cases after delete: 1250


8. MongoDB Aggregation Queries

In [136]:
# Aggregation 1: failed deliveries by zone

zone_aggregation = {}

for case in service_cases:
    zone = case.get("hub", {}).get("zone", "Unknown")
    failed = case["risk_flags"]["failed_delivery"]

    if zone not in zone_aggregation:
        zone_aggregation[zone] = {
            "total_cases": 0,
            "failed_cases": 0
        }

    zone_aggregation[zone]["total_cases"] += 1

    if failed:
        zone_aggregation[zone]["failed_cases"] += 1

zone_results = []

for zone, values in zone_aggregation.items():
    failure_rate = round(values["failed_cases"] / values["total_cases"] * 100, 2)
    zone_results.append({
        "zone": zone,
        "total_cases": values["total_cases"],
        "failed_cases": values["failed_cases"],
        "failure_rate_percent": failure_rate
    })

zone_results = sorted(zone_results, key=lambda x: x["failure_rate_percent"], reverse=True)

pprint(zone_results)

[{'failed_cases': 49,
  'failure_rate_percent': 20.16,
  'total_cases': 243,
  'zone': 'Central'},
 {'failed_cases': 15,
  'failure_rate_percent': 14.42,
  'total_cases': 104,
  'zone': 'Airport'},
 {'failed_cases': 16,
  'failure_rate_percent': 12.6,
  'total_cases': 127,
  'zone': 'West'},
 {'failed_cases': 17,
  'failure_rate_percent': 12.5,
  'total_cases': 136,
  'zone': 'North'},
 {'failed_cases': 14,
  'failure_rate_percent': 12.17,
  'total_cases': 115,
  'zone': 'Riverside'},
 {'failed_cases': 10,
  'failure_rate_percent': 9.43,
  'total_cases': 106,
  'zone': 'South'},
 {'failed_cases': 11,
  'failure_rate_percent': 9.24,
  'total_cases': 119,
  'zone': 'East'},
 {'failed_cases': 0,
  'failure_rate_percent': 0.0,
  'total_cases': 300,
  'zone': 'Unknown'}]


In [137]:
# Aggregation 2: complaints by service type

complaint_aggregation = {}

for case in service_cases:
    service_type = case.get("order", {}).get("service_type", "Unknown")
    complaints_list = case.get("complaints", [])

    if service_type not in complaint_aggregation:
        complaint_aggregation[service_type] = {
            "total_complaints": 0,
            "total_compensation": 0
        }

    complaint_aggregation[service_type]["total_complaints"] += len(complaints_list)

    for complaint in complaints_list:
        compensation = complaint.get("compensation_amount", 0)
        if compensation is not None:
            complaint_aggregation[service_type]["total_compensation"] += compensation

complaint_results = []

for service_type, values in complaint_aggregation.items():
    complaint_results.append({
        "service_type": service_type,
        "total_complaints": values["total_complaints"],
        "total_compensation": round(values["total_compensation"], 2)
    })

complaint_results = sorted(complaint_results, key=lambda x: x["total_complaints"], reverse=True)

pprint(complaint_results)

[{'service_type': 'Passenger',
  'total_compensation': nan,
  'total_complaints': 84},
 {'service_type': 'Retail', 'total_compensation': nan, 'total_complaints': 83},
 {'service_type': 'Parcel', 'total_compensation': nan, 'total_complaints': 77},
 {'service_type': 'Business',
  'total_compensation': 810.86,
  'total_complaints': 39},
 {'service_type': 'Medical', 'total_compensation': nan, 'total_complaints': 37}]


In [138]:
# Aggregation 3: route override impact

route_aggregation = {
    "Route override used": {
        "total_cases": 0,
        "failed_cases": 0,
        "ratings": [],
        "costs": []
    },
    "No route override": {
        "total_cases": 0,
        "failed_cases": 0,
        "ratings": [],
        "costs": []
    }
}

for case in service_cases:
    override_used = case["risk_flags"]["route_override_used"]
    group = "Route override used" if override_used else "No route override"

    route_aggregation[group]["total_cases"] += 1

    if case["risk_flags"]["failed_delivery"]:
        route_aggregation[group]["failed_cases"] += 1

    rating = case.get("delivery", {}).get("customer_rating_post_delivery")
    cost = case.get("delivery", {}).get("fuel_or_charge_cost")

    if rating is not None:
        route_aggregation[group]["ratings"].append(rating)

    if cost is not None:
        route_aggregation[group]["costs"].append(cost)

route_results = []

for group, values in route_aggregation.items():
    failure_rate = round(values["failed_cases"] / values["total_cases"] * 100, 2)
    avg_rating = round(sum(values["ratings"]) / len(values["ratings"]), 2)
    avg_cost = round(sum(values["costs"]) / len(values["costs"]), 2)

    route_results.append({
        "route_override_group": group,
        "total_cases": values["total_cases"],
        "failed_cases": values["failed_cases"],
        "failure_rate_percent": failure_rate,
        "avg_customer_rating": avg_rating,
        "avg_cost": avg_cost
    })

pprint(route_results)

[{'avg_cost': 13.14,
  'avg_customer_rating': nan,
  'failed_cases': 86,
  'failure_rate_percent': 15.61,
  'route_override_group': 'Route override used',
  'total_cases': 551},
 {'avg_cost': 12.43,
  'avg_customer_rating': nan,
  'failed_cases': 46,
  'failure_rate_percent': 6.58,
  'route_override_group': 'No route override',
  'total_cases': 699}]


 9. Indexing and Query Optimisation


In [139]:
# Recommended MongoDB indexes for the service_cases collection

recommended_indexes = [
    {
        "index": "order_id",
        "reason": "Fast lookup of one service case"
    },
    {
        "index": "customer.customer_id",
        "reason": "Find all service cases for a customer"
    },
    {
        "index": "hub.zone + risk_flags.failed_delivery",
        "reason": "Find failed services in a specific zone"
    },
    {
        "index": "order.service_type + risk_flags.has_complaint",
        "reason": "Analyse complaint-heavy service types"
    },
    {
        "index": "delivery.delivery_status + delivery.customer_rating_post_delivery",
        "reason": "Find failed or low-rated deliveries"
    },
    {
        "index": "app_events.event_type + app_events.success_flag",
        "reason": "Investigate failed app workflows"
    }
]

for item in recommended_indexes:
    print("Index:", item["index"])
    print("Reason:", item["reason"])
    print("-" * 60)

Index: order_id
Reason: Fast lookup of one service case
------------------------------------------------------------
Index: customer.customer_id
Reason: Find all service cases for a customer
------------------------------------------------------------
Index: hub.zone + risk_flags.failed_delivery
Reason: Find failed services in a specific zone
------------------------------------------------------------
Index: order.service_type + risk_flags.has_complaint
Reason: Analyse complaint-heavy service types
------------------------------------------------------------
Index: delivery.delivery_status + delivery.customer_rating_post_delivery
Reason: Find failed or low-rated deliveries
------------------------------------------------------------
Index: app_events.event_type + app_events.success_flag
Reason: Investigate failed app workflows
------------------------------------------------------------


In [140]:
# Example PyMongo index syntax
# This is demonstration syntax. Do not run against Atlas without a real connection string.

pymongo_index_examples = """
service_cases.create_index("order_id")
service_cases.create_index("customer.customer_id")
service_cases.create_index([("hub.zone", 1), ("risk_flags.failed_delivery", 1)])
service_cases.create_index([("order.service_type", 1), ("risk_flags.has_complaint", 1)])
service_cases.create_index([("delivery.delivery_status", 1), ("delivery.customer_rating_post_delivery", 1)])
service_cases.create_index([("app_events.event_type", 1), ("app_events.success_flag", 1)])

query = {"hub.zone": "Central", "risk_flags.failed_delivery": True}
service_cases.find(query).explain()
"""

print(pymongo_index_examples)


service_cases.create_index("order_id")
service_cases.create_index("customer.customer_id")
service_cases.create_index([("hub.zone", 1), ("risk_flags.failed_delivery", 1)])
service_cases.create_index([("order.service_type", 1), ("risk_flags.has_complaint", 1)])
service_cases.create_index([("delivery.delivery_status", 1), ("delivery.customer_rating_post_delivery", 1)])
service_cases.create_index([("app_events.event_type", 1), ("app_events.success_flag", 1)])

query = {"hub.zone": "Central", "risk_flags.failed_delivery": True}
service_cases.find(query).explain()

